# Phase 2: Transaction Cleaning

This notebook cleans the transaction dataset before any GIS merge or model work.

Scope:
- coerce numeric columns
- remove duplicate records
- remove invalid `market_value` and `Area` rows
- create `value_per_area` and `log_value_per_area`
- save cleaned output to `data/interim/`


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.src.config import configure_logging, ensure_directories, settings
from backend.src.data_cleaning import clean_transaction_data, save_cleaned_transactions
from backend.src.data_loader import load_transaction_data

configure_logging()
ensure_directories()
PROJECT_ROOT, settings


(WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc'),
 Settings(project_root=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc'), raw_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data'), interim_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/interim'), processed_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/processed'), reports_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports'), models_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/models'), transaction_file='tran_data.xlsx', property_shapefile='ai_usecase_data240326.shp', road_shapefile='ai_usecase_road240326.shp', facility_shapefile='ai_usecase_facilities240326.shp', log_level='INFO'))

In [2]:
transactions = load_transaction_data()
transactions.shape


2026-04-28 14:31:37,902 | INFO | src.data_loader | Loading transaction data from \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\data\tran_data.xlsx


(344079, 54)

In [3]:
transactions[['market_value', 'setforth_value', 'Area', 'Approach_Road_Width']].head()


,market_value,setforth_value,Area,Approach_Road_Width
0,900000.0,900000,5.729175,25.0
1,1215000.0,9000,9.000000,20.0
2,725000.0,725000,3.000000,12.0
3,504900.0,250000,3.300000,12.0
4,320000.0,320000,0.650000,12.0


In [4]:
cleaned_transactions, summary = clean_transaction_data(transactions)
cleaned_transactions.shape, summary


((273587, 56),
 CleaningSummary(input_row_count=344079, duplicate_rows_removed=70455, invalid_market_value_rows_removed=37, invalid_area_rows_removed=0, rows_with_value_per_area=273587, output_row_count=273587, missing_after_cleaning={'premises': 271790, 'Road_code': 9, 'Road_Name': 204125, 'Special_project_Name': 1929, 'Approach_Road_Width': 65, 'Adjacent_to_Metal_Road': 90454, 'Proposed_Land_use_Name': 90454, 'Nature_Land_use_Name': 90454, 'Proposed_Apartment_use_Name': 183133, 'Nature_Apartment_use_name': 183133, 'bargadar': 90454, 'whether_tenant_purchaser': 90454, 'is_bargadar_purchaser': 90454, 'Road_Category': 273587}))

In [5]:
cleaned_transactions[['market_value', 'Area', 'value_per_area', 'log_value_per_area']].head()


,market_value,Area,value_per_area,log_value_per_area
0,900000,5.729175,157090.680595,11.964585
1,1215000,9.0,135000.0,11.813037
2,725000,3.0,241666.666667,12.395319
3,504900,3.3,153000.0,11.9382
4,320000,0.65,492307.692308,13.106861


In [6]:
parquet_path, summary_path = save_cleaned_transactions(
    cleaned_transactions,
    summary,
    settings.interim_data_dir,
    settings.reports_dir,
)
parquet_path, summary_path


2026-04-28 14:38:39,466 | INFO | src.data_cleaning | Saved cleaned transactions to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\data\interim\cleaned_transactions.parquet
2026-04-28 14:38:39,467 | INFO | src.data_cleaning | Saved cleaning summary to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\reports\phase_2_transaction_cleaning_summary.json


(WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/interim/cleaned_transactions.parquet'),
 WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports/phase_2_transaction_cleaning_summary.json'))

In [7]:
json.loads(Path(summary_path).read_text())


{'input_row_count': 344079,
 'duplicate_rows_removed': 70455,
 'invalid_market_value_rows_removed': 37,
 'invalid_area_rows_removed': 0,
 'rows_with_value_per_area': 273587,
 'output_row_count': 273587,
 'missing_after_cleaning': {'premises': 271790,
  'Road_code': 9,
  'Road_Name': 204125,
  'Special_project_Name': 1929,
  'Approach_Road_Width': 65,
  'Adjacent_to_Metal_Road': 90454,
  'Proposed_Land_use_Name': 90454,
  'Nature_Land_use_Name': 90454,
  'Proposed_Apartment_use_Name': 183133,
  'Nature_Apartment_use_name': 183133,
  'bargadar': 90454,
  'whether_tenant_purchaser': 90454,
  'is_bargadar_purchaser': 90454,
  'Road_Category': 273587}}